# Merge trước → kiểm tra dữ liệu → loại dòng không hợp lệ

Notebook chạy trên **Google Colab**. Upload 5 file CSV:

- `gee_features.csv`
- `gee_terrain_features.csv`
- `macrostrat_features.csv`
- `nasa_power_features.csv`
- `gee_terrain_failed_sites.csv`

**Quy tắc:** Full outer join 4 bảng theo **3 cột đầu** (không dựa vào số thứ tự dòng). Sau khi merge xong mới xóa các site trong danh sách failed và mọi dòng có giá trị thiếu hoặc token không hợp lệ trong bất kỳ cột nào. Nếu 1 site chỉ có trong một vài file thì dòng đó cũng bị xóa ở bước lọc.

> `Unknown` mặc định được tính là không hợp lệ. Có thể chỉnh `INVALID_TEXT_VALUES` nếu `Unknown` là dữ liệu hợp lệ của nghiên cứu. Giá trị số `0` là hợp lệ.

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files
from pathlib import Path

DATA_FILES = [
    "gee_features.csv",
    "gee_terrain_features.csv",
    "macrostrat_features.csv",
    "nasa_power_features.csv",
]
FAILED_FILE = "gee_terrain_failed_sites.csv"
OUTPUT_FILE = "merged_earth_features_clean.csv"
REJECTED_FILE = "merged_rejected_sites.csv"
REPORT_FILE = "merge_quality_report.csv"

# Sửa danh sách sau để phù hợp với quy ước dữ liệu nghiên cứu của bạn.
INVALID_TEXT_VALUES = {
    "", "n/a", "na", "nan", "null", "none", "unknown", "missing",
    "no data", "nodata", "not available", "-9999", "-99999",
}

print("Upload 5 file CSV cùng lúc:")
uploaded = files.upload()
missing_files = [name for name in DATA_FILES + [FAILED_FILE] if not Path(name).is_file()]
if missing_files:
    raise FileNotFoundError(f"Thiếu file: {missing_files}")
print("Đã nhận đủ 5 file.")

In [ ]:
# Đọc toàn bộ file; giữ nguyên các chuỗi đặc biệt để tự kiểm tra sau khi merge.
tables = {}
for filename in DATA_FILES:
    df = pd.read_csv(filename, low_memory=False, keep_default_na=False)
    if df.shape[1] < 4:
        raise ValueError(f"{filename}: cần có ít nhất 3 cột khóa và 1 cột dữ liệu")
    tables[filename] = df

key_cols = list(tables[DATA_FILES[0]].columns[:3])
for filename, df in tables.items():
    if list(df.columns[:3]) != key_cols:
        raise ValueError(
            f"3 cột đầu trong {filename} không giống file đầu tiên. "
            f"Mong đợi: {key_cols}; tìm thấy: {list(df.columns[:3])}"
        )
    if df.duplicated(subset=key_cols).any():
        example = df.loc[df.duplicated(subset=key_cols, keep=False), key_cols].head(5)
        raise ValueError(f"{filename} có khóa site trùng nhau. Ví dụ:\n{example}")

failed = pd.read_csv(FAILED_FILE, low_memory=False, keep_default_na=False)
if list(failed.columns[:3]) != key_cols:
    raise ValueError(f"3 cột đầu của {FAILED_FILE} không khớp: {key_cols}")

print("Khóa ghép:", key_cols)
for filename, df in tables.items():
    print(f"{filename}: {len(df):,} dòng, {len(df.columns)} cột")
print(f"{FAILED_FILE}: {len(failed):,} dòng")

In [ ]:
# BƯỚC 1: MERGE TRƯỚC — chưa xóa dòng NA / dữ liệu lỗi.
# outer join giữ lại toàn bộ site xuất hiện trong bất kỳ file nào.
# Thêm tiền tố nếu nhiều file có cùng tên cột dữ liệu để tránh đè cột.
merged = None
source_markers = []
used_feature_names = set()

for index, filename in enumerate(DATA_FILES, start=1):
    df = tables[filename].copy()
    feature_cols = list(df.columns[3:])
    rename = {}
    for col in feature_cols:
        if col in used_feature_names:
            new_name = f"{Path(filename).stem}__{col}"
            while new_name in used_feature_names:
                new_name = "extra__" + new_name
            rename[col] = new_name
            used_feature_names.add(new_name)
        else:
            used_feature_names.add(col)
    df = df.rename(columns=rename)

    marker = f"__present_{index}__"
    df[marker] = True
    source_markers.append(marker)
    if merged is None:
        merged = df
    else:
        merged = merged.merge(df, how="outer", on=key_cols, sort=False, validate="one_to_one")

print(f"Sau khi MERGE (chưa lọc): {len(merged):,} dòng, {len(merged.columns) - len(source_markers)} cột thực tế")
display(merged.head())

In [ ]:
# BƯỚC 2: KIỂM TRA trên bảng ĐÃ MERGE.
# Loại nếu: site nằm trong failed_sites, thiếu trong >=1 file, hoặc có bất kỳ ô dữ liệu lỗi.

# So sánh failed bằng cả 3 cột khóa, không xóa trước merge.
failed_keys = pd.MultiIndex.from_frame(failed[key_cols].drop_duplicates())
merged_keys = pd.MultiIndex.from_frame(merged[key_cols])
in_failed_list = pd.Series(merged_keys.isin(failed_keys), index=merged.index)

missing_source = ~merged[source_markers].fillna(False).astype(bool).all(axis=1)
real_cols = [c for c in merged.columns if c not in source_markers]

# Không coi giá trị 0 hoặc False là sai. Chuỗi được trim và so khớp không phân biệt hoa/thường.
invalid_by_column = pd.DataFrame(index=merged.index)
for col in real_cols:
    s = merged[col]
    invalid = s.isna()
    invalid |= s.astype("string").str.strip().str.lower().isin(INVALID_TEXT_VALUES).fillna(False)
    if pd.api.types.is_numeric_dtype(s):
        invalid |= pd.Series(~np.isfinite(pd.to_numeric(s, errors="coerce")), index=s.index)
    invalid_by_column[col] = invalid.fillna(False)

invalid_cell = invalid_by_column.any(axis=1)
reject = in_failed_list | missing_source | invalid_cell

# Lý do từ chối (phục vụ audit).
rejected = merged.loc[reject, real_cols].copy()
rejected["reason_failed_sites"] = in_failed_list.loc[reject].to_numpy()
rejected["reason_missing_from_source"] = missing_source.loc[reject].to_numpy()
rejected["reason_invalid_cell"] = invalid_cell.loc[reject].to_numpy()
rejected["invalid_columns"] = invalid_by_column.loc[reject].apply(
    lambda row: ", ".join(row.index[row]), axis=1
)

clean = merged.loc[~reject, real_cols].reset_index(drop=True)
print(f"Sau merge:                 {len(merged):,} dòng")
print(f"Site thuộc failed_sites:   {int(in_failed_list.sum()):,} dòng")
print(f"Thiếu trong >=1 file:      {int(missing_source.sum()):,} dòng")
print(f"Có ô dữ liệu không hợp lệ: {int(invalid_cell.sum()):,} dòng")
print(f"Tổng bị xóa (không cộng chồng): {int(reject.sum()):,} dòng")
print(f"CÒN LẠI:                  {len(clean):,} dòng")
print("Lưu ý: một dòng có thể thỏa nhiều lý do bị xóa.")
display(clean.head())

In [ ]:
# BƯỚC 3: KIỂM TRA KẾT QUẢ & TẢI VỀ.
assert len(clean) + len(rejected) == len(merged)
assert not clean.isna().any().any()
assert not clean.duplicated(subset=key_cols).any()

report = pd.DataFrame([
    {"metric": "input_rows_" + fn, "count": len(tables[fn])} for fn in DATA_FILES
] + [
    {"metric": "rows_after_outer_merge", "count": len(merged)},
    {"metric": "rows_in_failed_site_list", "count": int(in_failed_list.sum())},
    {"metric": "rows_missing_from_one_or_more_sources", "count": int(missing_source.sum())},
    {"metric": "rows_with_invalid_cells", "count": int(invalid_cell.sum())},
    {"metric": "rows_rejected_unique", "count": int(reject.sum())},
    {"metric": "rows_final_clean", "count": len(clean)},
])

clean.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
rejected.to_csv(REJECTED_FILE, index=False, encoding="utf-8-sig")
report.to_csv(REPORT_FILE, index=False, encoding="utf-8-sig")

for name in [OUTPUT_FILE, REJECTED_FILE, REPORT_FILE]:
    files.download(name)
print("Đã xuất 3 file: merged sạch, dòng bị loại + lý do, báo cáo tổng hợp.")

### Ghi chú quan trọng

- **Gộp trước, lọc sau**, đúng trình tự yêu cầu. `gee_terrain_failed_sites.csv` chỉ là danh sách site cần loại, không phải bảng đặc trưng để ghép.
- Dùng **full outer join theo 3 khóa**, thay vì ghép theo vị trí dòng: tránh làm sai site khi một file thiếu dòng.
- Dòng có bất kỳ thuộc tính thiếu hoặc `Unknown` sẽ bị loại (*complete-case filtering*). Điều này có thể làm giảm mạnh dữ liệu; nếu cần giữ lại hoặc nội suy thiếu, phải thay đổi quy tắc.
- Nếu site có 3 khóa trùng trong cùng một file, notebook sẽ dừng để tránh phép join nhân bản dữ liệu.
- File rất lớn có thể vượt RAM Colab khi full outer merge; khi đó cần quy trình chunking/SQL/DuckDB.